
## Folder 04 — File 08: Silver Incremental Merge:


- Use `Delta Lake MERGE` to **insert new records** and **update existing records without overwriting** the entire Silver table.
- This file demonstrates an important Data Engineering concept: **Upsert** (UPDATE + INSERT).


### How incremental MERGE works?


```text
┌──────────────────────────────────────────────┐
│             INCOMING SOURCE RECORDS          │
│                                              │
│       New and Previously Existing Records    │
└──────────────────────────────────────────────┘
                       │
                       ▼
┌──────────────────────────────────────────────┐
│                DELTA LAKE MERGE              │
│                                              │
│       Match records using Business Keys      │
└──────────────────────────────────────────────┘
                       │
              ┌────────┴────────┐
              ▼                 ▼
┌──────────────────────┐  ┌──────────────────────┐
│       MATCHED        │  │     NOT MATCHED      │
│                      │  │                      │
│ UPDATE existing      │  │ INSERT new records  │
│ records              │  │                      │
└──────────────────────┘  └──────────────────────┘
              │                 │
              └────────┬────────┘
                       ▼
┌──────────────────────────────────────────────┐
│          UPDATED SILVER DELTA TABLES         │
│                                              │
│       Deduplicated and Updated Records       │
└──────────────────────────────────────────────┘
```


## Business Keys:


Business keys are `unique identifiers` used to match `source records` with `existing target records` during **Delta Lake MERGE** operations.


| Dataset | MERGE Key |
|---|---|
| Customers | `customer_id` |
| Products | `product_id` |
| Orders | `order_id` |
| Order Items | `order_item_id` |
| Payments | `payment_id` |
| Shipments | `shipment_id` |
| Inventory | `product_id` + `warehouse_id` + `snapshot_date` |

In [0]:
## 1. Import libraries

from pyspark.sql import functions as F
from delta.tables import DeltaTable

print("Libraries imported successfully.")


## 2. Project configuration

CATALOG = "workspace"
SILVER_SCHEMA = "ecommerce_silver"

# Dataset configuration:
# table name -> business key columns

MERGE_CONFIG = {
    "silver_customers": ["customer_id"],
    "silver_products": ["product_id"],
    "silver_orders": ["order_id"],
    "silver_order_items": ["order_item_id"],
    "silver_payments": ["payment_id"],
    "silver_shipments": ["shipment_id"],
    "silver_inventory": [
        "product_id",
        "warehouse_id",
        "snapshot_date"
    ]
}


 ## 3. Create reusable MERGE function

def merge_to_silver(
    source_df,
    target_table,
    key_columns
):

    target_exists = spark.catalog.tableExists(target_table)

    if not target_exists:
        raise Exception(
            f"Target Silver table does not exist: {target_table}"
        )

    if not key_columns:
        raise ValueError("At least one merge key is required.")

    # Verify that all merge keys exist in the source.

    missing_keys = [
        key for key in key_columns
        if key not in source_df.columns
    ]

    if missing_keys:
        raise Exception(
            f"Missing merge keys in source: {missing_keys}"
        )

    # Reject null business keys.

    null_key_condition = None

    for key in key_columns:
        condition = F.col(key).isNull()

        null_key_condition = (
            condition
            if null_key_condition is None
            else null_key_condition | condition
        )

    source_df = source_df.filter(~null_key_condition)

    # Prevent multiple source rows from matching

    # the same target row.

    duplicate_source_keys = (
        source_df
        .groupBy(*key_columns)
        .count()
        .filter(F.col("count") > 1)
        .limit(1)
        .count()
    )

    if duplicate_source_keys > 0:
        raise Exception(
            f"Source contains duplicate merge keys for {target_table}. "
            "Deduplicate before MERGE."
        )

    delta_target = DeltaTable.forName(
        spark,
        target_table
    )

    merge_condition = " AND ".join(
        [
            f"target.`{key}` <=> source.`{key}`"
            for key in key_columns
        ]
    )

    # Update only columns that exist in both tables.

    # Preserve target columns that are absent in source.

    target_columns = spark.table(target_table).columns

    common_columns = [
        column
        for column in target_columns
        if column in source_df.columns
    ]

    update_columns = {
        column: f"source.`{column}`"
        for column in common_columns
        if column not in key_columns
    }

    insert_columns = {
        column: f"source.`{column}`"
        for column in common_columns
    }

    (
        delta_target.alias("target")
        .merge(
            source_df.alias("source"),
            merge_condition
        )
        .whenMatchedUpdate(
            set=update_columns
        )
        .whenNotMatchedInsert(
            values=insert_columns
        )
        .execute()
    )

    print(f"MERGE completed: {target_table}")


## 4. Read existing Silver datasets

silver_sources = {}

for table_name in MERGE_CONFIG.keys():

    full_table_name = (
        f"{CATALOG}.{SILVER_SCHEMA}.{table_name}"
    )

    if not spark.catalog.tableExists(full_table_name):
        raise Exception(
            f"Silver table not found: {full_table_name}. "
            "Run the previous Silver transformation notebooks."
        )

    silver_sources[table_name] = spark.table(full_table_name)

    print(
        f"{table_name}: "
        f"{silver_sources[table_name].count()} records"
    )


## 5. Execute incremental MERGE

for table_name, key_columns in MERGE_CONFIG.items():

    target_table = (
        f"{CATALOG}.{SILVER_SCHEMA}.{table_name}"
    )

    source_df = silver_sources[table_name]

    merge_to_silver(
        source_df=source_df,
        target_table=target_table,
        key_columns=key_columns
    )


## 6. Validate merged tables

merge_validation_results = []

for table_name, key_columns in MERGE_CONFIG.items():

    target_table = (
        f"{CATALOG}.{SILVER_SCHEMA}.{table_name}"
    )

    target_df = spark.table(target_table)

    record_count = target_df.count()

    duplicate_groups = (
        target_df
        .groupBy(*key_columns)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    null_key_condition = None

    for key in key_columns:

        condition = F.col(key).isNull()

        null_key_condition = (
            condition
            if null_key_condition is None
            else null_key_condition | condition
        )

    null_key_count = (
        target_df
        .filter(null_key_condition)
        .count()
    )

    merge_validation_results.append(
        (
            table_name,
            record_count,
            duplicate_groups,
            null_key_count
        )
    )


## 7. Display MERGE validation report

validation_df = spark.createDataFrame(
    merge_validation_results,
    [
        "table_name",
        "record_count",
        "duplicate_key_groups",
        "null_key_records"
    ]
)

display(validation_df)


## 8. Fail pipeline if validation fails

failed_tables = (
    validation_df
    .filter(
        (F.col("duplicate_key_groups") > 0)
        | (F.col("null_key_records") > 0)
    )
    .collect()
)

if failed_tables:

    raise Exception(
        f"MERGE validation failed: {failed_tables}"
    )

print("All Silver MERGE validations passed.")


## 9. Display Delta table history

history_table = (
    f"{CATALOG}.{SILVER_SCHEMA}.silver_orders"
)

display(
    DeltaTable.forName(
        spark,
        history_table
    ).history()
)


## 10. Final summary

print("=" * 60)
print("SILVER INCREMENTAL MERGE COMPLETED")
print("=" * 60)

for table_name in MERGE_CONFIG:

    target_table = (
        f"{CATALOG}.{SILVER_SCHEMA}.{table_name}"
    )

    print(
        f"{table_name}: "
        f"{spark.table(target_table).count()} records"
    )

print("=" * 60)




## Data Processing Operations:


| Operation | What it does |
|---|---|
| **Full Refresh** | Reprocesses all records and overwrites the target table. |
| **Upsert / MERGE** | Updates matching keys and inserts new keys. |
| **Incremental Ingestion** | Processes only newly arrived or changed records. |
| **CDC (Change Data Capture)** | Captures source INSERT, UPDATE, and DELETE operations. |
